In [1]:
import polars as pl
import glob

In [2]:
train_files = sorted(glob.glob('../data/train_part_*.parquet'))

operations = pl.scan_parquet(train_files)
labels = pl.scan_parquet('../data/train_labels.parquet')

operations = operations.with_columns(
    pl.col('event_dttm').str.to_datetime(),
    pl.col('event_type_nm').cast(pl.Int8, strict=False),
    pl.col('event_desc').cast(pl.Int16, strict=False),
    pl.col('channel_indicator_type').cast(pl.Int8, strict=False),
    pl.col('channel_indicator_sub_type').cast(pl.Int16, strict=False),
    pl.col('operaton_amt').cast(pl.Float32, strict=False),
    pl.col('currency_iso_cd').cast(pl.Int16, strict=False),
    pl.col('pos_cd').cast(pl.Int16, strict=False),
    pl.col('timezone').cast(pl.Int16, strict=False),
    pl.col('operating_system_type').cast(pl.Int8, strict=False),
    pl.col('developer_tools').cast(pl.Int8, strict=False),
    pl.col('phone_voip_call_state').cast(pl.Int8, strict=False),
    pl.col('web_rdp_connection').cast(pl.Int8, strict=False),
    pl.col('battery').str.replace('%', '').cast(pl.Int8, strict=False),
    pl.col('mcc_code').cast(pl.Int8, strict=False),
    pl.col('screen_size').str.split_exact('x', 1)
    .struct.rename_fields(['screen_w', 'screen_h'])
    .alias('screen_dims')
)

operations = operations.with_columns(
    pl.col('event_dttm').dt.date().alias('event_date')
).unnest('screen_dims').with_columns(
    pl.col('screen_w').cast(pl.Int16, strict=False),
    pl.col('screen_h').cast(pl.Int16, strict=False)
)
operations = operations.drop(['accept_language', 'browser_language', 'screen_size', 'device_system_version'])


labels = labels.with_columns(
    pl.col('target').cast(pl.Int8)
)

df = operations.join(labels, on=['customer_id', 'event_id'], how='left')
df = df.with_columns(pl.col('target').fill_null(0))

In [3]:
df.head().collect()

customer_id,event_id,event_dttm,event_type_nm,event_desc,channel_indicator_type,channel_indicator_sub_type,operaton_amt,currency_iso_cd,mcc_code,pos_cd,timezone,session_id,operating_system_type,battery,developer_tools,phone_voip_call_state,web_rdp_connection,compromised,screen_w,screen_h,event_date,target
i64,i64,datetime[μs],i8,i16,i8,i16,f32,i16,i8,i16,i16,i64,i8,i8,i8,i8,i8,str,i16,i16,date,i8
123123123123129,123999300382879,2024-10-01 05:29:14,14,75,6,5,56422.0,0,4,3,null,null,null,null,null,null,null,null,null,null,2024-10-01,0
123123123123129,124531875713936,2024-10-01 10:17:22,7,56,4,15,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2024-10-01,0
123123123123129,123329285580171,2024-10-01 10:20:03,3,120,6,5,300870.0,0,10,3,null,null,null,null,null,null,null,null,null,null,2024-10-01,0
123123123123129,124334305430665,2024-10-02 07:48:09,14,75,6,5,298458.0,0,1,3,null,null,null,null,null,null,null,null,null,null,2024-10-02,0
123123123123129,126215501146513,2024-10-02 11:20:40,14,75,6,5,59944.0,0,15,3,null,null,null,null,null,null,null,null,null,null,2024-10-02,0


In [6]:
df.collect().shape

(85677840, 23)

In [3]:
df.select(pl.col('target').value_counts()).collect()

target
struct[2]
"{0,85626402}"
"{1,51438}"


In [4]:
df.collect_schema()
df.collect().n_unique('customer_id')

100000

In [5]:
df.collect().estimated_size('mb')

5327.564253807068

In [8]:
df.sink_parquet('../data/train_full.parquet')